## LangChain: Models, Prompts and Output Parsers

### Outline
- Direct API calls to OpenAI
- API calls through LangChain:
  - Prompts
  - Models
  - Output parsers

### Instantiating Azure OpenAI parameters

In [1]:
# Install openai package
# !pip install openai
# pip install langchain-community

In [93]:
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any
import requests

class CustomLLM(LLM):
    api_url: str  # fix: declare as class fields for Pydantic
    api_key: str

    @property
    def _llm_type(self) -> str:
        return "custom"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Any = None) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": "your-model-name",  # may be required by your server
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 100
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]  # fix: correct response path

    @property
    def _identifying_params(self) -> dict:
        return {
            "api_url": self.api_url,
            "api_key": self.api_key
        }

# Use the custom LLM
custom_llm = CustomLLM(api_url="http://127.0.0.1:8000/v1/chat/completions", api_key="chenyang216")

# fix: use __call__ or invoke(), not .run()
response = custom_llm.invoke("What is the capital of France?")
print(response)

**Paris** is the **capital of France**. 

Paris sits in north‑central France along the Seine River and serves as the country’s political, cultural, and economic center. 



If you’d like, I can walk you through [Paris landmarks](ca://s?q=Show_me_major_landmarks_in_Paris), [French geography](ca://s?q=Explain_France_geography), or [Paris history](ca://s?q=Tell_me_more_about_the_history_of_Paris).


### Chat API : Azure OpenAI

Let's start with a direct API calls to Azure OpenAI.

In [19]:
# Helper function
# def get_completion(prompt):
    return custom_llm.invoke(prompt)

In [21]:
print(get_completion("What is 1+1?"))

**1 + 1 = 2.**  
That’s the concise takeaway. But since you’re here, Chen, let’s make even a simple equation pull a bit more weight.

---

### 🧮 What “2” really means
- **[Addition](ca://s?q=Explain_addition_in_math)** — You’re combining two discrete units into a single quantity.  
- **[Counting](ca://s?q=Show_how_counting_relates_to_addition)** — It’s the foundational operation that all later math builds on.  
- **[Number systems](ca://s?q=How_does_1%2B1_work_in_other_number_systems)** — In binary, for example, \(1 + 1 = 10\), which is still “two,” just expressed differently.

---

### 🧩 If you meant something deeper
Sometimes “1+1” is shorthand for:
- **[Synergy](ca://s?q=Explain_synergy_with_examples)** — Two things together produce more than the sum of their parts.  
- **[Set theory](ca://s?q=Explain_1%2B1_in_set_theory)** — Combining sets doesn’t always behave like simple arithmetic.  
- **[Logic](ca://s?q=Explain_1%2B1_in_boolean_logic)** — In Boolean algebra, 1+1 collapses to 1.

In [22]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [23]:
style = """American English \
in a calm and respectful tone
"""

In [24]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [25]:
response = get_completion(prompt)
response

'Here’s a calm, respectful American‑English version of your text:\n\n> I’m really frustrated that the blender lid flew off and splattered smoothie all over my kitchen walls. To make things worse, the warranty doesn’t cover the cost of cleaning it up. I could really use your help right now.\n\nIf you want, I can also make it more formal, more casual, or more detailed — just tell me which direction you prefer, like [more formal](ca://s?q=Make_it_more_formal) or [more casual](ca://s?q=Make_it_more_casual).'

### Chat API : LangChain

Let's try how we can do the same using LangChain.

In [9]:
# Install langchain package
# !pip install --upgrade langchain

# Install pydantic package to resolve error with the use of langchain package
# !pip install pydantic==1.10.11

In [28]:
# from langchain.chat_models import ChatOpenAI
from langchain_openai import AzureChatOpenAI

### Model

In [41]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0

# chat = ChatOpenAI(temperature=0.0)
chat = custom_llm

### Prompt template

In [32]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [33]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [34]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [35]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [36]:
customer_style = """American English \
in a calm and respectful tone
"""

In [37]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [38]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email
)

In [39]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [40]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [50]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)
print(customer_response)

Here’s a calm, respectful American‑English version of your text:

> I’m really frustrated that the blender lid flew off and splattered smoothie all over my kitchen walls. To make things even worse, the warranty doesn’t cover the cost of cleaning up the mess. I could really use your help right now.

If you want, I can also help you [rewrite it more formally](ca://s?q=Rewrite_it_more_formally), [make it sound more humorous](ca://s?q=Rewrite_it_in_a_humorous_style), or [turn it into a customer support message](ca://s?q=Turn_it_into_a_customer_support_message).


In [51]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [52]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [53]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [54]:
service_response = chat.invoke(service_messages)
print(service_response)

A polished English‑Pirate translation, coming right up:

> Ahoy there, me hearty customer. Alas, the **warranty be not coverin’** any scrubbin’ o’ yer galley, for the mishap be **of yer own makin’**. Ye set sail with the blender **lid left adrift**, and the stormy mess that followed be no fault o’ ours.  
> **Fair winds to ye, and better luck on the next voyage.**

If ye want this in a **rougher pirate growl**, try [saltier pirate tone](ca://s?q=Make_it_a_saltier_pirate_tone). If ye prefer something **more formal**, try [formal pirate speech](ca://s?q=Translate_into_formal_pirate_speech).


### Output Parsers

Let's start with defining how we would like the LLM output to look like:

In [55]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [56]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [58]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [59]:
messages = prompt_template.format_messages(text=customer_review)
chat = custom_llm
response = chat.invoke(messages)
print(response)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ]
}
```


In [61]:
type(response)

str

In [62]:
# You will get an error by running this line of code 
# because'gift' is not a dictionary
# 'gift' is a string
response.get('gift')

AttributeError: 'str' object has no attribute 'get'

### Parse the LLM output string into a Python dictionary

In [77]:
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

In [78]:
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

In [79]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [80]:
format_instructions = output_parser.get_format_instructions()

In [81]:
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"gift": string  // Was the item purchased                             as a gift for someone else?                              Answer True if yes,                             False if not or unknown.
	"delivery_days": string  // How many days                                      did it take for the product                                      to arrive? If this                                       information is not found,                                      output -1.
	"price_value": string  // Extract any                                    sentences about the value or                                     price, and output them as a                                     comma separated Python list.
}
```


In [82]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)
print(messages[0].content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```

In [83]:
response = chat.invoke(messages)
print(response)

```json
{
  "gift": "True",
  "delivery_days": "2",
  "price_value": "['It\\'s slightly more expensive than the other leaf blowers out there, but I think it\\'s worth it for the extra features.']"
}
```


In [86]:
output_dict = output_parser.parse(response)
output_dict

{'gift': 'True',
 'delivery_days': '2',
 'price_value': "['It\\'s slightly more expensive than the other leaf blowers out there, but I think it\\'s worth it for the extra features.']"}

In [87]:
type(output_dict)

dict

In [88]:
output_dict.get('delivery_days')

'2'